# Intervention Helpers

Demonstrates selectors, activation transforms, and backward hook helpers.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.log_forward_pass(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.log_forward_pass(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.log_forward_pass(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "ModelLog": log,
    "LayerLog": layer_log,
    "LayerPassLog": layer_pass,
    "ModuleLog": module_log,
    "ModulePassLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnPassLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Intervention Helpers: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.bwd_hook",
    "tl.clamp",
    "tl.contains",
    "tl.func",
    "tl.gradient_scale",
    "tl.gradient_zero",
    "tl.in_module",
    "tl.intervention.AppendBatchDependenceError",
    "tl.intervention.AppendMismatchError",
    "tl.intervention.ArgComponent",
    "tl.intervention.AxisAmbiguityError",
    "tl.intervention.BaselineUndeterminedError",
    "tl.intervention.BatchNormTrainModeWarning",
    "tl.intervention.Bundle",
    "tl.intervention.BundleMemberError",
    "tl.intervention.BundleRelationshipError",
    "tl.intervention.CapturedArgTemplate",
    "tl.intervention.ContainerSpec",
    "tl.intervention.ControlFlowDivergenceError",
    "tl.intervention.ControlFlowDivergenceWarning",
    "tl.intervention.DataclassField",
    "tl.intervention.DeadParentError",
    "tl.intervention.DictKey",
    "tl.intervention.DirectActivationWriteWarning",
    "tl.intervention.DirectWriteInExecutableSaveError",
    "tl.intervention.EdgeUseRecord",
    "tl.intervention.EngineDispatchError",
    "tl.intervention.FireRecord",
    "tl.intervention.ForkFieldPolicy",
    "tl.intervention.FrozenInterventionSpec",
    "tl.intervention.FrozenTargetSpec",
    "tl.intervention.FunctionRegistryKey",
    "tl.intervention.GraphShapeMismatchError",
    "tl.intervention.HFKey",
    "tl.intervention.HelperSpec",
    "tl.intervention.HookContext",
    "tl.intervention.HookHandle",
    "tl.intervention.HookSignatureError",
    "tl.intervention.HookSiteCoverageError",
    "tl.intervention.HookValueError",
    "tl.intervention.InterventionReadyConflictError",
    "tl.intervention.InterventionSpec",
    "tl.intervention.LiteralTensor",
    "tl.intervention.LiteralValue",
    "tl.intervention.LiveModeLabelError",
    "tl.intervention.ModelMismatchError",
    "tl.intervention.MultiMatchWarning",
    "tl.intervention.MutateInPlaceWarning",
    "tl.intervention.NamedField",
    "tl.intervention.NoParentError",
    "tl.intervention.NormalizedHookEntry",
    "tl.intervention.OpaqueCallableInExecutableSaveError",
    "tl.intervention.ParentRef",
    "tl.intervention.RecursiveTracingError",
    "tl.intervention.Relationship",
    "tl.intervention.ReplayPreconditionError",
    "tl.intervention.SaveLevel",
    "tl.intervention.SiteAmbiguityError",
    "tl.intervention.SiteCollection",
    "tl.intervention.SiteResolutionError",
    "tl.intervention.SiteSpec",
    "tl.intervention.SiteTable",
    "tl.intervention.SpecCompat",
    "tl.intervention.SpecMutationError",
    "tl.intervention.SpecPortabilityError",
    "tl.intervention.SpliceModuleDeviceError",
    "tl.intervention.SpliceModuleDtypeError",
    "tl.intervention.TargetManifestDiff",
    "tl.intervention.TargetSpec",
    "tl.intervention.TensorSliceSpec",
    "tl.intervention.TupleIndex",
    "tl.intervention.Unsupported",
    "tl.intervention.bwd_hook",
    "tl.intervention.check_spec_compat",
    "tl.intervention.clamp",
    "tl.intervention.contains",
    "tl.intervention.do",
    "tl.intervention.func",
    "tl.intervention.gradient_scale",
    "tl.intervention.gradient_zero",
    "tl.intervention.in_module",
    "tl.intervention.label",
    "tl.intervention.load_intervention_spec",
    "tl.intervention.make_hook_context",
    "tl.intervention.mean_ablate",
    "tl.intervention.module",
    "tl.intervention.noise",
    "tl.intervention.normalize_hook",
    "tl.intervention.normalize_hook_plan",
    "tl.intervention.project_off",
    "tl.intervention.project_onto",
    "tl.intervention.replay",
    "tl.intervention.replay_from",
    "tl.intervention.rerun",
    "tl.intervention.resample_ablate",
    "tl.intervention.resolve_sites",
    "tl.intervention.save_intervention",
    "tl.intervention.scale",
    "tl.intervention.sites",
    "tl.intervention.splice_module",
    "tl.intervention.steer",
    "tl.intervention.swap_with",
    "tl.intervention.where",
    "tl.intervention.zero_ablate",
    "tl.label",
    "tl.mean_ablate",
    "tl.module",
    "tl.noise",
    "tl.project_off",
    "tl.project_onto",
    "tl.resample_ablate",
    "tl.scale",
    "tl.splice_module",
    "tl.steer",
    "tl.swap_with",
    "tl.where",
    "tl.zero_ablate",
]

selectors = [
    tl.label("linear_1_1"),
    tl.func("linear"),
    tl.contains("linear"),
    tl.module("0"),
    tl.where(lambda layer: "linear" in layer.layer_label),
    tl.in_module("0"),
]
for selector in selectors:
    print(type(selector).__name__, summarize_value(log.find_sites(selector)))
direction = torch.ones_like(layer_pass.activation)
transforms = [
    tl.clamp(min=0.0),
    tl.scale(0.5),
    tl.zero_ablate(),
    tl.noise(0.01, seed=0),
    tl.mean_ablate(),
    tl.project_off(direction),
    tl.project_onto(direction),
    tl.resample_ablate(source=clean),
    tl.steer(direction, magnitude=0.1),
    tl.swap_with("linear_1_1"),
    tl.splice_module(nn.Identity()),
]
print("transforms", [type(transform).__name__ for transform in transforms])
print("backward", type(tl.gradient_zero()).__name__, type(tl.gradient_scale(0.5)).__name__)

## Intervention Helpers coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.bwd_hook",
    "tl.clamp",
    "tl.contains",
    "tl.func",
    "tl.gradient_scale",
    "tl.gradient_zero",
    "tl.in_module",
    "tl.intervention.AppendBatchDependenceError",
    "tl.intervention.AppendMismatchError",
    "tl.intervention.ArgComponent",
    "tl.intervention.AxisAmbiguityError",
    "tl.intervention.BaselineUndeterminedError",
    "tl.intervention.BatchNormTrainModeWarning",
    "tl.intervention.Bundle",
    "tl.intervention.BundleMemberError",
    "tl.intervention.BundleRelationshipError",
    "tl.intervention.CapturedArgTemplate",
    "tl.intervention.ContainerSpec",
    "tl.intervention.ControlFlowDivergenceError",
    "tl.intervention.ControlFlowDivergenceWarning",
    "tl.intervention.DataclassField",
    "tl.intervention.DeadParentError",
    "tl.intervention.DictKey",
    "tl.intervention.DirectActivationWriteWarning",
    "tl.intervention.DirectWriteInExecutableSaveError",
    "tl.intervention.EdgeUseRecord",
    "tl.intervention.EngineDispatchError",
    "tl.intervention.FireRecord",
    "tl.intervention.ForkFieldPolicy",
    "tl.intervention.FrozenInterventionSpec",
    "tl.intervention.FrozenTargetSpec",
    "tl.intervention.FunctionRegistryKey",
]

audit_touch_items("Intervention Helpers coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Intervention Helpers coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.intervention.GraphShapeMismatchError",
    "tl.intervention.HFKey",
    "tl.intervention.HelperSpec",
    "tl.intervention.HookContext",
    "tl.intervention.HookHandle",
    "tl.intervention.HookSignatureError",
    "tl.intervention.HookSiteCoverageError",
    "tl.intervention.HookValueError",
    "tl.intervention.InterventionReadyConflictError",
    "tl.intervention.InterventionSpec",
    "tl.intervention.LiteralTensor",
    "tl.intervention.LiteralValue",
    "tl.intervention.LiveModeLabelError",
    "tl.intervention.ModelMismatchError",
    "tl.intervention.MultiMatchWarning",
    "tl.intervention.MutateInPlaceWarning",
    "tl.intervention.NamedField",
    "tl.intervention.NoParentError",
    "tl.intervention.NormalizedHookEntry",
    "tl.intervention.OpaqueCallableInExecutableSaveError",
    "tl.intervention.ParentRef",
    "tl.intervention.RecursiveTracingError",
    "tl.intervention.Relationship",
    "tl.intervention.ReplayPreconditionError",
    "tl.intervention.SaveLevel",
    "tl.intervention.SiteAmbiguityError",
    "tl.intervention.SiteCollection",
    "tl.intervention.SiteResolutionError",
    "tl.intervention.SiteSpec",
    "tl.intervention.SiteTable",
    "tl.intervention.SpecCompat",
    "tl.intervention.SpecMutationError",
]

audit_touch_items("Intervention Helpers coverage cluster 2", ITEMS, AUDIT_CONTEXT)

## Intervention Helpers coverage cluster 3

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.intervention.SpecPortabilityError",
    "tl.intervention.SpliceModuleDeviceError",
    "tl.intervention.SpliceModuleDtypeError",
    "tl.intervention.TargetManifestDiff",
    "tl.intervention.TargetSpec",
    "tl.intervention.TensorSliceSpec",
    "tl.intervention.TupleIndex",
    "tl.intervention.Unsupported",
    "tl.intervention.bwd_hook",
    "tl.intervention.check_spec_compat",
    "tl.intervention.clamp",
    "tl.intervention.contains",
    "tl.intervention.do",
    "tl.intervention.func",
    "tl.intervention.gradient_scale",
    "tl.intervention.gradient_zero",
    "tl.intervention.in_module",
    "tl.intervention.label",
    "tl.intervention.load_intervention_spec",
    "tl.intervention.make_hook_context",
    "tl.intervention.mean_ablate",
    "tl.intervention.module",
    "tl.intervention.noise",
    "tl.intervention.normalize_hook",
    "tl.intervention.normalize_hook_plan",
    "tl.intervention.project_off",
    "tl.intervention.project_onto",
    "tl.intervention.replay",
    "tl.intervention.replay_from",
    "tl.intervention.rerun",
    "tl.intervention.resample_ablate",
    "tl.intervention.resolve_sites",
]

audit_touch_items("Intervention Helpers coverage cluster 3", ITEMS, AUDIT_CONTEXT)

## Intervention Helpers coverage cluster 4

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.intervention.save_intervention",
    "tl.intervention.scale",
    "tl.intervention.sites",
    "tl.intervention.splice_module",
    "tl.intervention.steer",
    "tl.intervention.swap_with",
    "tl.intervention.where",
    "tl.intervention.zero_ablate",
    "tl.label",
    "tl.mean_ablate",
    "tl.module",
    "tl.noise",
    "tl.project_off",
    "tl.project_onto",
    "tl.resample_ablate",
    "tl.scale",
    "tl.splice_module",
    "tl.steer",
    "tl.swap_with",
    "tl.where",
    "tl.zero_ablate",
]

audit_touch_items("Intervention Helpers coverage cluster 4", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.bwd_hook",
    "tl.clamp",
    "tl.contains",
    "tl.func",
    "tl.gradient_scale",
    "tl.gradient_zero",
    "tl.in_module",
    "tl.intervention.AppendBatchDependenceError",
    "tl.intervention.AppendMismatchError",
    "tl.intervention.ArgComponent",
    "tl.intervention.AxisAmbiguityError",
    "tl.intervention.BaselineUndeterminedError",
    "tl.intervention.BatchNormTrainModeWarning",
    "tl.intervention.Bundle",
    "tl.intervention.BundleMemberError",
    "tl.intervention.BundleRelationshipError",
    "tl.intervention.CapturedArgTemplate",
    "tl.intervention.ContainerSpec",
    "tl.intervention.ControlFlowDivergenceError",
    "tl.intervention.ControlFlowDivergenceWarning",
    "tl.intervention.DataclassField",
    "tl.intervention.DeadParentError",
    "tl.intervention.DictKey",
    "tl.intervention.DirectActivationWriteWarning",
    "tl.intervention.DirectWriteInExecutableSaveError",
    "tl.intervention.EdgeUseRecord",
    "tl.intervention.EngineDispatchError",
    "tl.intervention.FireRecord",
    "tl.intervention.ForkFieldPolicy",
    "tl.intervention.FrozenInterventionSpec",
    "tl.intervention.FrozenTargetSpec",
    "tl.intervention.FunctionRegistryKey",
    "tl.intervention.GraphShapeMismatchError",
    "tl.intervention.HFKey",
    "tl.intervention.HelperSpec",
    "tl.intervention.HookContext",
    "tl.intervention.HookHandle",
    "tl.intervention.HookSignatureError",
    "tl.intervention.HookSiteCoverageError",
    "tl.intervention.HookValueError",
    "tl.intervention.InterventionReadyConflictError",
    "tl.intervention.InterventionSpec",
    "tl.intervention.LiteralTensor",
    "tl.intervention.LiteralValue",
    "tl.intervention.LiveModeLabelError",
    "tl.intervention.ModelMismatchError",
    "tl.intervention.MultiMatchWarning",
    "tl.intervention.MutateInPlaceWarning",
    "tl.intervention.NamedField",
    "tl.intervention.NoParentError",
    "tl.intervention.NormalizedHookEntry",
    "tl.intervention.OpaqueCallableInExecutableSaveError",
    "tl.intervention.ParentRef",
    "tl.intervention.RecursiveTracingError",
    "tl.intervention.Relationship",
    "tl.intervention.ReplayPreconditionError",
    "tl.intervention.SaveLevel",
    "tl.intervention.SiteAmbiguityError",
    "tl.intervention.SiteCollection",
    "tl.intervention.SiteResolutionError",
    "tl.intervention.SiteSpec",
    "tl.intervention.SiteTable",
    "tl.intervention.SpecCompat",
    "tl.intervention.SpecMutationError",
    "tl.intervention.SpecPortabilityError",
    "tl.intervention.SpliceModuleDeviceError",
    "tl.intervention.SpliceModuleDtypeError",
    "tl.intervention.TargetManifestDiff",
    "tl.intervention.TargetSpec",
    "tl.intervention.TensorSliceSpec",
    "tl.intervention.TupleIndex",
    "tl.intervention.Unsupported",
    "tl.intervention.bwd_hook",
    "tl.intervention.check_spec_compat",
    "tl.intervention.clamp",
    "tl.intervention.contains",
    "tl.intervention.do",
    "tl.intervention.func",
    "tl.intervention.gradient_scale",
    "tl.intervention.gradient_zero",
    "tl.intervention.in_module",
    "tl.intervention.label",
    "tl.intervention.load_intervention_spec",
    "tl.intervention.make_hook_context",
    "tl.intervention.mean_ablate",
    "tl.intervention.module",
    "tl.intervention.noise",
    "tl.intervention.normalize_hook",
    "tl.intervention.normalize_hook_plan",
    "tl.intervention.project_off",
    "tl.intervention.project_onto",
    "tl.intervention.replay",
    "tl.intervention.replay_from",
    "tl.intervention.rerun",
    "tl.intervention.resample_ablate",
    "tl.intervention.resolve_sites",
    "tl.intervention.save_intervention",
    "tl.intervention.scale",
    "tl.intervention.sites",
    "tl.intervention.splice_module",
    "tl.intervention.steer",
    "tl.intervention.swap_with",
    "tl.intervention.where",
    "tl.intervention.zero_ablate",
    "tl.label",
    "tl.mean_ablate",
    "tl.module",
    "tl.noise",
    "tl.project_off",
    "tl.project_onto",
    "tl.resample_ablate",
    "tl.scale",
    "tl.splice_module",
    "tl.steer",
    "tl.swap_with",
    "tl.where",
    "tl.zero_ablate",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")